# Machine Learning
### Multinomial naïve Bayes

**Multinomial naïve Bayes** (**multinomial NB**) is a probabilistic machine learning **classifier** based on **Bag-of-Words** (BoW) representation of feacture vectors, and **multinomial distribution** assumption.

- It is widely used for **text classification** problems — such as spam detection, sentiment analysis, topic categorization, language detection — where the input is a **bag-of-words** (i.e., word frequencies), not individual word presence.
- Unlike **Bernoulli** NB (which models binary features: word present/absent), Multinomial NB models count features: how many times each word appears.

Thus, multinomial NB assumes:
- **Bag-of-words**: Each feature $x_i$ is the number of $word_i$ that appears in the given document
- **Multinomial distirbution**: It assumes that given the class $C=k$, the feature vector (count-vector) $\boldsymbol(x)=(x_1,x_2,...,x_q)$ of a document (sample) follows the multinomial distribution:
  - $P(x_1,x_2,...,x_q|C=k)=\frac{(\sum_{i=1}^q x_i)!}{\prod_{i=1}^q x_i !}\prod_{i=1}^{q}P(x_i|C=k)$ 
    - where $P(x_i|C=k)=P(word_i|C=k)^{x_i}$
<hr>

Let:
- $C =$ class label (e.g., “spam”, “not spam”)
- $\boldsymbol{x} =$ feature vector (e.g., word counts: $[count(word_1), count(word_2), ..., count(word_q)]$
  - Thus, $x_i=count(word_i)$: is the number of $word_i$ in the given sample (document). 
- $q =$ total vocabulary size (number of unique words across all classes)
<hr>

Multinomial NB is composed of two algorithms or phases: **Training** and **prediction**:
- **Training algorithm**: Given feature matrix $X$ (each row holds features of one sample) and corresponding class labels $\boldsymbol{y}$, we compute prior probailities $P(C=k)$ and likelihood of each feature (word) given the class, which is denoted by $P(word_i|C=k)$ or $P_{i|k}$:
  - $P(C=k)=\frac{\text{number of samples with } C=k}{\text{total number of samples}}$
  - $P(word_i|C=k)=P_{i|k}=\frac{count(word_i \text{ in class } k)+1}{\text{total words in class } k +q}$
    - We notice that for $x_i=count(word_i)$: $P(x_i|C=k)=P_{i|k}^{x_i}$
   - As we said above, $q=\text{total unique words across all classes}$
- **Prediction algorithm**: For the new feature vector $\boldsymbol{x}_{new}$, we choose the class with maximum **posterior**:
  - $C_{predicted}=arg max_k P(C=k|\boldsymbol{x}_{new})$
  - where $P(C=k|\boldsymbol{x}_{new})\propto P(\boldsymbol{x}_{new}|C=k)\cdot P(C=k)$
  - such that $P(\boldsymbol{x}_{new}|C=k)\propto\prod_{i=1}^q P_{i|k}^{x_i}$ 
- In summary: $P(C=k|\boldsymbol{x}_{new})\propto P(C=k)\cdot \prod_{i=1}^q P_{i|k}^{x_i}$
- For numerical stability, we use $log$ such that: $C_{predicted}=arg max_k log P(C=k|\boldsymbol{x}_{new})$ 
<hr>

**Hint:** Some points on Multinomial NB:
- Multinomial NB uses **bag-of-words** representation.
  - Cannot directly handle continuous or negative features.
- Works well with high dimensional sparse data.
- Sensitive to irrelant or highly correlated features.
- Extremly fast to train and predict.

<hr>

**Reminder:** We have computed here the class posterior probability by Bayes\' theorem:
<br> $P(C=k|\boldsymbol{x})=\frac{P(\boldsymbol{x}|C=k) \cdot P(C=k)}{P(\boldsymbol{x})}$
<br> Since the denominator $P(\boldsymbol{x})$ is constant for all classes, we ignore it for classification, and only compare the numerators in the prediction algorithm.
<hr>


In the following:
- We implement **Multinomial NB** from scratch in Python. 
- Then, we create a synthetic corpus of  five documents belonging to one of the two classes: *Sports*, and *Politics*. 
- Finally, our Multinomial NB is used for **document classification** based on the given corpus.

<hr>

https://github.com/ostad-ai/Machine-Learning
<br> Explanation: https://www.pinterest.com/HamedShahHosseini/Machine-Learning/

In [1]:
# Import required modules
import math
from collections import defaultdict

In [2]:
class MultinomialNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = set()
        
        # Count of DOCUMENTS per class (used for Priors)
        self.doc_counts = defaultdict(int)  
        
        # Count of WORDS per class (used for Likelihood denominator)
        self.word_counts_per_class = defaultdict(int) 
        
        # Frequency of each specific word per class
        self.word_counts = defaultdict(lambda: defaultdict(int))  
        
        self.vocab = set()
        self.priors = {}

    def fit(self, X, y):
        """
        X: list of lists of words (e.g., [['football', 'soccer'], ['election']])
        y: list of labels
       
        OPTIMIZATION: We do this in a SINGLE pass O(N)
        """
        self.classes = set(y)
        total_docs = len(X)
        
        # 1. Single pass to count everything
        for doc, label in zip(X, y):
            self.doc_counts[label] += 1  # Count the document
            
            for word in doc:
                self.word_counts[label][word] += 1
                self.word_counts_per_class[label] += 1 # Count the word
                self.vocab.add(word)
                
        self.vocab_size = len(self.vocab)
        
        # 2. Compute priors based on DOCUMENT counts
        for class_label in self.classes:
            self.priors[class_label] = self.doc_counts[class_label] / total_docs

    def predict_proba(self, X):
        """
        Returns normalized probability for each class for each sample.
        Uses Log Probabilities to prevent numerical underflow.
        """
        results = []
        for sample in X:
            log_probs = {}
            
            for class_label in self.classes:
                # Start with the log of the prior probability
                # (Add a tiny epsilon to prevent log(0) if a prior is somehow 0)
                log_prob = math.log(self.priors[class_label] + 1e-10)
                
                total_words_in_class = self.word_counts_per_class[class_label]
                
                # Add the log likelihood of each word in the sample
                for word in sample:
                    count = self.word_counts[class_label].get(word, 0)
                    
                    # Calculate smoothed probability
                    prob = (count + self.alpha) / (total_words_in_class + self.alpha * self.vocab_size)
                    
                    # Add log(prob) instead of multiplying prob
                    log_prob += math.log(prob + 1e-10)
                    
                log_probs[class_label] = log_prob
            
            # Convert log probabilities back to actual probabilities safely
            # We subtract the max log prob to prevent overflow when exponentiating
            max_log = max(log_probs.values())
            probs = {k: math.exp(v - max_log) for k, v in log_probs.items()}
            
            # Normalize so they sum to 1.0
            total_prob = sum(probs.values())
            normalized_probs = {k: v / total_prob for k, v in probs.items()}
            
            results.append(normalized_probs)
            
        return results

    def predict(self, X):
        """
        Return predicted class for each sample
        """
        probas = self.predict_proba(X)
        # Find the class with the highest probability
        return [max(proba.items(), key=lambda x: x[1])[0] for proba in probas]

In [3]:
# Example usage
X = [
    ["football", "soccer"],
    ["basketball"],
    ["election", "vote"],
    ["football", "soccer"],
    ["vote"]]
y = ["Sports", "Sports", "Politics", "Sports", "Politics"]

nb = MultinomialNaiveBayes(alpha=1.0)
nb.fit(X, y)

# Predict for new samples
test_samples = [
    ["football", "soccer"],
    ["election", "vote", "vote"]]

predictions = nb.predict(test_samples)
print("Predictions:", predictions)

# Get probabilities
probas = nb.predict_proba(test_samples)
for i, prob in enumerate(probas):
    print(f"Sample {i}: {prob}")

Predictions: ['Sports', 'Politics']
Sample 0: {'Politics': 0.10373443992854815, 'Sports': 0.8962655600714519}
Sample 1: {'Politics': 0.9590792838096297, 'Sports': 0.040920716190370285}
